# lora-scan results
Run `python scripts/aggregate.py` first (or use `collect_results` directly, as below).

In [ ]:
import pandas as pd
from lorascan import runs
df = pd.DataFrame(runs.collect_results())
df[["experiment_id","seed","method","location","accuracy","n","mean_completion_tokens"]].sort_values(["experiment_id","seed"])

In [ ]:
# Accuracy per experiment, aggregated over seeds
summary = df.groupby("experiment_id")["accuracy"].agg(["mean","std","count"]).sort_values("mean", ascending=False)
summary

In [ ]:
import yaml
from scipy.stats import binomtest  # noqa
from statsmodels.stats.contingency_tables import mcnemar

registry = yaml.safe_load(open(runs.ROOT / "predictions" / "registry.yaml"))

def correct_by_item(experiment_id, seed):
    rows = runs.load_predictions(runs.RUNS, registry[experiment_id][seed])
    return {r["item_id"]: r["correct"] for r in rows}

def paired_mcnemar(exp_a, exp_b, seed):
    a, b = correct_by_item(exp_a, seed), correct_by_item(exp_b, seed)
    ids = sorted(set(a) & set(b))
    both = sum(a[i] and b[i] for i in ids); only_a = sum(a[i] and not b[i] for i in ids)
    only_b = sum(b[i] and not a[i] for i in ids); neither = len(ids) - both - only_a - only_b
    res = mcnemar([[both, only_a], [only_b, neither]], exact=True)
    return {"n": len(ids), "only_a": only_a, "only_b": only_b, "p": res.pvalue}

# Example: did late beat random on the same items?
# paired_mcnemar("qwen15_gsm8k_grpo_late", "qwen15_gsm8k_grpo_random", seed=42)